# Face → Height / Weight / BMI Estimator (Colab-ready)

This notebook works straight from the **full project zip** you already have
(`Face-to-height-weight-BMI-estimation--master.zip`) — it contains the labels CSV,
230 training photos, and 26 test photos, all in one file.

**All you do:** run every cell top to bottom. Cell 2 will ask you to upload
**one file** — that zip. Everything else (unzip, find the CSV, find the photos,
train, save, test) happens automatically.

## 1. Install dependencies (modern-Colab compatible)

In [ ]:
!pip install -q face_recognition scikit-learn joblib pandas numpy pillow


## 2. Upload the project zip
Upload **Face-to-height-weight-BMI-estimation--master.zip** (the whole repo zip) when the file picker appears. Just that one file.

In [ ]:
import os, zipfile, glob
from google.colab import files

print("Upload the project zip file (Face-to-height-weight-BMI-estimation--master.zip)")
uploaded = files.upload()

EXTRACT_DIR = "project_data"
os.makedirs(EXTRACT_DIR, exist_ok=True)

for fname in uploaded:
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(EXTRACT_DIR)
        print(f"Extracted {fname} -> ./{EXTRACT_DIR}/")
    else:
        print(f"Skipping {fname} (not a zip) — if this was meant to be the dataset zip, re-upload just the .zip file")


## 3. Auto-locate the CSV and the photo folders
No need to know the exact folder name inside the zip — this searches for them.

In [ ]:
# find the labels CSV anywhere inside the extracted zip
csv_candidates = glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True)
assert csv_candidates, "No CSV found inside the zip — check the upload."
label_file = csv_candidates[0]
print("Using labels file:", label_file)

# find the training photos folder (named 'height_weight', not 'height_weight_test')
folder_candidates = [
    d for d in glob.glob(f"{EXTRACT_DIR}/**/height_weight", recursive=True)
    if os.path.isdir(d)
]
assert folder_candidates, "No 'height_weight' photo folder found inside the zip."
data_folder = folder_candidates[0]
print("Using training photos folder:", data_folder)

# optional: the held-out test photos folder, if present
test_candidates = [
    d for d in glob.glob(f"{EXTRACT_DIR}/**/height_weight_test", recursive=True)
    if os.path.isdir(d)
]
test_folder = test_candidates[0] if test_candidates else None
print("Using test photos folder:", test_folder)


## 4. Load labels and match them to photos

In [ ]:
import pandas as pd
from pathlib import Path as p
import re

profile_df = pd.read_csv(label_file)
print(f"Loaded {len(profile_df)} labeled people")
profile_df.head()


In [ ]:
all_files = glob.glob(data_folder + "/*")
all_jpgs = sorted([f for f in all_files if f.lower().endswith((".jpg", ".jpeg", ".png"))])
print(f"Found {len(all_jpgs)} training photos in {data_folder}")

def get_index_of_digit(filename_stem):
    match = re.search(r"\d", filename_stem)
    return match.start(0) if match else len(filename_stem)

# UID is everything in the filename before the first digit, e.g. "akshay1.jpg" -> "akshay"
id_path = [(p(img).stem[:get_index_of_digit(p(img).stem)], img) for img in all_jpgs]
image_df = pd.DataFrame(id_path, columns=["UID", "path"])

data_df = image_df.merge(profile_df, on="UID")
print(f"Matched {len(data_df)} photos to labels")
data_df.head()


## 5. Extract a face embedding from every photo
Each face is turned into a 128-number vector using `face_recognition`'s pretrained face model. That vector is the input feature for our regressors. With 200+ photos this will take a few minutes on Colab's CPU.

In [ ]:
import face_recognition
import numpy as np

def get_face_encoding(image_path):
    image = face_recognition.load_image_file(image_path)
    encodings = face_recognition.face_encodings(image)
    if len(encodings) == 0:
        return None
    return encodings[0]

encodings, keep_idx = [], []
for i, row in data_df.reset_index(drop=True).iterrows():
    enc = get_face_encoding(row["path"])
    if enc is not None:
        encodings.append(enc)
        keep_idx.append(i)
    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(data_df)} photos...")

skipped = len(data_df) - len(keep_idx)
data_df = data_df.reset_index(drop=True).iloc[keep_idx].reset_index(drop=True)
X = np.array(encodings)
print(f"Built feature matrix: {X.shape[0]} samples x {X.shape[1]} features ({skipped} photos skipped — no face detected)")


## 6. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

y_height = data_df.height.values
y_weight = data_df.weight.values
y_bmi = data_df.BMI.values

(X_train, X_test,
 yh_train, yh_test,
 yw_train, yw_test,
 ybmi_train, ybmi_test) = train_test_split(
    X, y_height, y_weight, y_bmi, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")


## 7. Fit metric helper

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

def report_goodness(model, X_test, y_test, log_target=True):
    y_pred = model.predict(X_test)
    if log_target:
        y_pred = np.exp(y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"MSE: {mse:.4f}   R^2: {r2:.4f}")
    return mse, r2


## 8. Train the models
Kernel Ridge Regression performed best in the original experiments, so we train that directly (with a StandardScaler, which is the modern replacement for the old deprecated `normalize=True` option).

In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def train_model(X_train, y_train):
    model = make_pipeline(
        StandardScaler(),
        KernelRidge(kernel="rbf", gamma=0.21, alpha=0.0017)
    )
    model.fit(X_train, np.log(y_train))
    return model

print("Training height model...")
model_height = train_model(X_train, yh_train)
report_goodness(model_height, X_test, yh_test)

print("\nTraining weight model...")
model_weight = train_model(X_train, yw_train)
report_goodness(model_weight, X_test, yw_test)

print("\nTraining BMI model...")
model_bmi = train_model(X_train, ybmi_train)
report_goodness(model_bmi, X_test, ybmi_test)


## 9. Save the trained models

In [ ]:
import joblib

joblib.dump(model_height, "height_predictor.model")
joblib.dump(model_weight, "weight_predictor.model")
joblib.dump(model_bmi, "bmi_predictor.model")
print("Saved height_predictor.model, weight_predictor.model, bmi_predictor.model")


## 10. Try predictions on the held-out test photos

In [ ]:
def predict_height_weight_bmi(image_path, height_model, weight_model, bmi_model):
    enc = get_face_encoding(image_path)
    if enc is None:
        return None
    X_new = np.expand_dims(enc, axis=0)
    height = float(np.exp(height_model.predict(X_new)[0]))
    weight = float(np.exp(weight_model.predict(X_new)[0]))
    bmi = float(np.exp(bmi_model.predict(X_new)[0]))
    return {"height_m": round(height, 2), "weight_kg": round(weight, 1), "bmi": round(bmi, 1)}

test_images = sorted(glob.glob(test_folder + "/*")) if test_folder else all_jpgs[:5]
for img_path in test_images[:5]:
    result = predict_height_weight_bmi(img_path, model_height, model_weight, model_bmi)
    print(img_path, "->", result)


## 11. Download the models (to push to GitHub)

In [ ]:
from google.colab import files as colab_files

for f in ["height_predictor.model", "weight_predictor.model", "bmi_predictor.model"]:
    colab_files.download(f)


## 12. Push to GitHub
In a Colab cell (or your terminal):

```
!git clone https://github.com/<your-username>/<your-repo>.git
%cd <your-repo>
!cp /content/height_predictor.model /content/weight_predictor.model /content/bmi_predictor.model .
!git add height_predictor.model weight_predictor.model bmi_predictor.model
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
!git commit -m "Add trained height/weight/BMI models"
!git push
```
(You'll need a GitHub personal access token as the password when it prompts for auth.)

## 13. Deploy on Streamlit
Use the `app.py`, `requirements.txt`, and `packages.txt` provided alongside this notebook — put them in the root of the same GitHub repo, then go to https://share.streamlit.io, connect the repo, and deploy. First build takes 10-20 min because `dlib` compiles from source using `packages.txt` — that's normal and only happens once.